# Centered model

### Searchlight tests

#### Full loop

In [ ]:
# Import libraries and set parameters

import os
import glob
import numpy as np
from itertools import product
from variograd_utils import dataset, save_hdf5
from variograd_utils.loadsave_utils import load_predictions_arrays
from variograd_utils.test_utils import compare_errors
from variograd_utils.brain_utils import vertex_info_10k as vinfo
import warnings
warnings.filterwarnings("ignore")

grads = [1, 2, 3]
scales = [10, 50, 200]
hs = ["L", "R"]
side = 4
mdl_name = "centered"
stat_test = "ttest_rel"
alternative = "less"

train = dataset("train")
test = dataset("test")

for grad, scale, h in product(grads, scales, hs):
    try:
        pattern = f"{train.output_dir}/LKresults.S{scale}.G{grad}/side{side}/{h}/LKresults_hdf5_G{grad}_sl*.h5"
        h5_files = glob.glob(pattern)

        respath = train.outpath("results")
        if not os.path.exists(respath): os.makedirs(respath)

        # Load prediction arrays and compute errors

        # print(f"\n\nFound {len(h5_files)} files matching {pattern}")
        args = {"filepaths":h5_files, "group": "test", "n_subjects": test.N,
        "n_vertex": vinfo[f"gray{h.lower()}"].size, "n_threads": 10}
        test_obs, test_pred, test_var = load_predictions_arrays(**args, model_name = f"{mdl_name}")
        _, avg_pred, avg_var = load_predictions_arrays(**args, model_name = f"{mdl_name}_avg")
        _, null_pred, null_var = load_predictions_arrays(**args, model_name = f"{mdl_name}_null")


        # print(f"\n\nTest data shape: {test_obs.shape}, {test_pred.shape}, {avg_pred.shape}, {null_pred.shape}")

        fname = test.outpath(f"{test.id}.{h}.FC_embeddings.a05_t0.G{grad}.csv")
        full_gradients = np.loadtxt(fname, delimiter=",")

        test_means = full_gradients - test_obs

        subj_IDs = np.repeat(np.arange(test.N), test_obs.shape[1]).reshape(test_obs.shape)

        vtx_IDs = np.tile(np.arange(test_obs.shape[1]), test.N).reshape(test_obs.shape)

        fname = test.outpath(f"test.{h}.SL_IDs.JE_cauchy{scale}_l{side}.npy")
        sl_affiliation = np.load(fname)
        sl_IDs = np.unique(sl_affiliation)

        # print(f"\n\nFound {len(sl_IDs)} unique SLs in {fname}")


        # Compare errors across different models

        Mod_Avg = {}
        Mod_Means = {}
        Mod_Null = {}
        MeanMod_MeanAvg = {}
        MeanMod_MeanNull = {}
        Means_MeanNull = {}
        Means_MeanAvg = {}

        for i, sl in enumerate(sl_IDs):
            subjs = subj_IDs[sl_affiliation == sl]
            subjs_unq, n_obs = np.unique(subjs, return_counts=True)
            mask = np.isin(subjs, subjs_unq[n_obs > 1])
            
            subjs = subjs[mask]
            full = full_gradients[sl_affiliation == sl][mask]
            true = test_obs[sl_affiliation == sl][mask]
            if np.isnan(true).all(): continue
            pred = test_pred[sl_affiliation == sl][mask]
            avg = avg_pred[sl_affiliation == sl][mask]
            null = null_pred[sl_affiliation == sl][mask]
            means = test_means[sl_affiliation == sl][mask]

            args = {"stat_test": stat_test, "alternative": alternative, "groupby": subjs}
            # Test against group avg
            Mod_Avg[sl] = compare_errors(true, pred, avg, **args)
            # Test against means
            Mod_Means[sl] = compare_errors(full, means + pred, means, **args)
            # Test against mean+avg
            MeanMod_MeanAvg[sl] = compare_errors(full, means + pred, means + avg, **args)
            # Test against null
            Mod_Null[sl] = compare_errors(true, pred, null, **args)
            # Tests against means + null
            MeanMod_MeanNull[sl] = compare_errors(full, means + pred, (means + null.T).T, **args)
            # test if adding avg worsens means prediction
            Means_MeanAvg[sl] = compare_errors(full, means, means + avg, **args)
            # test if adding null worsens means prediction
            Means_MeanNull[sl] = compare_errors(full, means, (means + null.T).T, **args)

        # print("\n\nDone computing statistics.")

        resdict = {
            "Mod_Avg": Mod_Avg,
            "Mod_Means": Mod_Means,
            "Mod_Null": Mod_Null,
            "MeanMod_MeanAvg": MeanMod_MeanAvg,
            "MeanMod_MeanNull": MeanMod_MeanNull,
            "Means_MeanNull": Means_MeanNull,
            "Means_MeanAvg": Means_MeanAvg,
        }

        for name, res in resdict.items():
            fname = train.outpath(f"results/SLresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{name}.h5")
            save_hdf5(d=res, filename=fname)

        # print(f"\n\nSaved results to {respath}\n")
        print(f"\n\nCompleted analysis for G{grad}, S{scale}, {h}\n")

    except Exception as e:
        print(f"Error processing G{grad}, S{scale}, {h}:\n\t{e}")


#### Code blocks

In [ ]:
# Import libraries and set parameters

import os
import glob
import numpy as np
from variograd_utils import dataset, save_hdf5
from variograd_utils.loadsave_utils import load_predictions_arrays
from variograd_utils.test_utils import compare_errors
from variograd_utils.brain_utils import vertex_info_10k as vinfo

grad = 2
scale = 10
h = "R"
side = 4
mdl_name = "centered"
stat_test = "ttest_rel"
alternative = "less"

train = dataset("train")
test = dataset("test")
pattern = f"{train.output_dir}/LKresults.S{scale}.G{grad}/side{side}/{h}/LKresults_hdf5_G{grad}_sl*.h5"
h5_files = glob.glob(pattern)

respath = train.outpath("results")
if not os.path.exists(respath): os.makedirs(respath)

# Load prediction arrays and compute errors

print(f"\n\nFound {len(h5_files)} files matching {pattern}")

args = {"filepaths":h5_files, "group": "test", "n_subjects": test.N,
        "n_vertex": vinfo[f"gray{h.lower()}"].size, "n_threads": 10}
test_obs, test_pred, test_var = load_predictions_arrays(**args, model_name = f"{mdl_name}")
_, avg_pred, avg_var = load_predictions_arrays(**args, model_name = f"{mdl_name}_avg")
_, null_pred, null_var = load_predictions_arrays(**args, model_name = f"{mdl_name}_null")

print(f"\n\nTest data shape: {test_obs.shape}, {test_pred.shape}, {avg_pred.shape}, {null_pred.shape}")

fname = test.outpath(f"{test.id}.{h}.FC_embeddings.a05_t0.G{grad}.csv")
full_gradients = np.loadtxt(fname, delimiter=",")
test_means = full_gradients - test_obs
subj_IDs = np.repeat(np.arange(test.N), test_obs.shape[1]).reshape(test_obs.shape)
vtx_IDs = np.tile(np.arange(test_obs.shape[1]), test.N).reshape(test_obs.shape)

fname = test.outpath(f"test.{h}.SL_IDs.JE_cauchy{scale}_l{side}.npy")
sl_affiliation = np.load(fname)
sl_IDs = np.unique(sl_affiliation)

print(f"\n\nFound {len(sl_IDs)} unique SLs in {fname}")


In [ ]:
# Compare errors across different models

Mod_Avg = {}
Mod_Means = {}
Mod_Null = {}
MeanMod_MeanAvg = {}
MeanMod_MeanNull = {}
Means_MeanNull = {}
Means_MeanAvg = {}

for i, sl in enumerate(sl_IDs):
    subjs = subj_IDs[sl_affiliation == sl]
    subjs_unq, n_obs = np.unique(subjs, return_counts=True)
    mask = np.isin(subjs, subjs_unq[n_obs > 1])
    
    subjs = subjs[mask]
    full = full_gradients[sl_affiliation == sl][mask]
    true = test_obs[sl_affiliation == sl][mask]
    if np.isnan(true).all(): continue
    pred = test_pred[sl_affiliation == sl][mask]
    avg = avg_pred[sl_affiliation == sl][mask]
    null = null_pred[sl_affiliation == sl][mask]
    means = test_means[sl_affiliation == sl][mask]

    args = {"stat_test": stat_test, "alternative": alternative, "groupby": subjs}
    # Test against group avg
    Mod_Avg[sl] = compare_errors(true, pred, avg, **args)
    # Test against means
    Mod_Means[sl] = compare_errors(full, means + pred, means, **args)
    # Test against mean+avg
    MeanMod_MeanAvg[sl] = compare_errors(full, means + pred, means + avg, **args)
    # Test against null
    Mod_Null[sl] = compare_errors(true, pred, null, **args)
    # Tests against means + null
    MeanMod_MeanNull[sl] = compare_errors(full, means + pred, (means + null.T).T, **args)
    # test if adding avg worsens means prediction
    Means_MeanAvg[sl] = compare_errors(full, means, means + avg, **args)
    # test if adding null worsens means prediction
    Means_MeanNull[sl] = compare_errors(full, means, (means + null.T).T, **args)

print("\n\nDone computing statistics.")

resdict = {
    "Mod_Avg": Mod_Avg,
    "Mod_Means": Mod_Means,
    "Mod_Null": Mod_Null,
    "MeanMod_MeanAvg": MeanMod_MeanAvg,
    "MeanMod_MeanNull": MeanMod_MeanNull,
    "Means_MeanNull": Means_MeanNull,
    "Means_MeanAvg": Means_MeanAvg,
}

for name, res in resdict.items():
    fname = train.outpath(f"results/SLresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{name}.h5")
    save_hdf5(d=res, filename=fname)

print(f"\n\nSaved results to {respath}\n")

### Vertex tests

#### Full loop

In [ ]:
import os
import glob
import warnings
import numpy as np
from itertools import product
from statsmodels.stats.multitest import multipletests
import variograd_utils as vu
from variograd_utils.test_utils import compare_errors
from importlib import reload
reload(vu)
warnings.filterwarnings("ignore")


grads = [1, 2, 3]
scales = [10, 50, 200]
hs = ["L", "R"]
side = 4
mdl_name = "centered"
stat_test = "ttest_rel"
alternative = "less"


train = vu.dataset("train")
test = vu.dataset("test")

for grad, scale, h in product(grads, scales, hs):
    try:
        print("\n\nProcessing G{}, S{}, {}\n".format(grad, scale, h))

        pattern = f"{train.output_dir}/LKresults.S{scale}.G{grad}/side{side}/{h}/LKresults_hdf5_G{grad}_sl*.h5"
        h5_files = glob.glob(pattern)

        respath = train.outpath("results")
        if not os.path.exists(respath): os.makedirs(respath)


        # Load prediction arrays
        args = {"filepaths":h5_files, "group": "test", "n_subjects": test.N,
        "n_vertex": vu.vertex_info_10k[f"gray{h.lower()}"].size, "n_threads": 10}
        test_obs, test_pred, test_var = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}")
        _, avg_pred, avg_var = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}_avg")
        _, null_pred, _ = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}_null")

        print(f"\tTest data shape: {test_obs.shape}, {test_pred.shape}, {avg_pred.shape}, {null_pred.shape}")


        fname = test.outpath(f"{test.id}.{h}.FC_embeddings.a05_t0.G{grad}.csv")
        full_gradients = np.loadtxt(fname, delimiter=",")
        test_means = full_gradients - test_obs
        subj_IDs = np.repeat(np.arange(test.N), test_obs.shape[1]).reshape(test_obs.shape)
        vtx_IDs = np.tile(np.arange(test_obs.shape[1]), test.N).reshape(test_obs.shape)

        fname = test.outpath(f"test.{h}.SL_IDs.JE_cauchy{scale}_l{side}.npy")
        sl_affiliation = np.load(fname)
        sl_IDs = np.unique(sl_affiliation)

        print(f"\tLoaded data for G{grad}, S{scale}, {h} with {len(sl_IDs)} unique SLs.")


        # Compare errors between models
        args = {"stat_test": stat_test,
                "alternative": alternative,
                # "weights": 1 - test_var / np.nanmax(test_var)
                "weights": 1 / np.exp(test_var)
        }

        # Test against means
        Mod_Means = compare_errors(full_gradients, test_means + test_pred, test_means, **args)
        # Test against mean+avg
        MeanMod_MeanAvg = compare_errors(full_gradients, test_means + test_pred, test_means + avg_pred, **args)
        # test if adding avg worsens means prediction
        Means_MeanAvg = compare_errors(full_gradients, test_means, test_means + avg_pred, **args)

        print("\tDone computing statistics for Means and Avg models.")

        # Test against null
        args["weights"] = np.atleast_3d(args["weights"])
        args["agg_errors"] = "mean"
        # Tests against means + null
        MeanMod_MeanNull = compare_errors(np.atleast_3d(full_gradients), np.atleast_3d(test_means + test_pred), np.atleast_3d(test_means) + null_pred, **args)
        # test if adding null worsens means prediction
        Means_MeanNull = compare_errors(np.atleast_3d(full_gradients), np.atleast_3d(test_means), np.atleast_3d(test_means) + null_pred, **args)

        print("\tDone computing statistics for Null models.")


        # correct for fdr
        p_unadj_all = np.hstack([Mod_Means["pval"], MeanMod_MeanAvg["pval"], Means_MeanAvg["pval"],
                                 MeanMod_MeanNull["pval"], Means_MeanNull["pval"]])

        p_adj_all = multipletests(p_unadj_all, method="fdr_bh")[1]
        p_adj_all = np.split(p_adj_all, 5)

        Mod_Means["pval_adj"] = p_adj_all[0]
        MeanMod_MeanAvg["pval_adj"] = p_adj_all[1]
        Means_MeanAvg["pval_adj"] = p_adj_all[2]
        MeanMod_MeanNull["pval_adj"] = p_adj_all[3]
        Means_MeanNull["pval_adj"] = p_adj_all[4]

        print("\tDone correcting p-values for multiple comparisons.")

        resdict = {
            "Mod_Means": Mod_Means,
            "MeanMod_MeanAvg": MeanMod_MeanAvg,
            "Means_MeanAvg": Means_MeanAvg,
            "MeanMod_MeanNull": MeanMod_MeanNull,
            "Means_MeanNull": Means_MeanNull,
        }
        for name, res in resdict.items():
            fname = train.outpath(f"results/VTXresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{name}.h5")
            vu.save_hdf5(d=res, filename=fname)
        print(f"\tCompleted analysis for G{grad}, S{scale}, {h}\n")

    except Exception as e:
        print(f"\t\tError processing G{grad}, S{scale}, {h}:\n\t{e}")



Processing G1, S50, R

	Test data shape: (241, 9398), (241, 9398), (241, 9398), (241, 9398, 100)
	Loaded data for G1, S50, R with 992 unique SLs.
	Done computing statistics for Means and Avg models.
	Done computing statistics for Null models.
	Done correcting p-values for multiple comparisons.
	Completed analysis for G1, S50, R



#### Code blocks

In [ ]:
# Import libraries and set parameters
import os
import glob
import warnings
import numpy as np
import variograd_utils as vu
warnings.filterwarnings("ignore")

grad = 1
scale = 50
h = "R"
side = 4
mdl_name = "centered"
stat_test = "ttest_rel"
alternative = "less"


train = vu.dataset("train")
test = vu.dataset("test")
pattern = f"{train.output_dir}/LKresults.S{scale}.G{grad}/side{side}/{h}/LKresults_hdf5_G{grad}_sl*.h5"
h5_files = glob.glob(pattern)

respath = train.outpath("results")
if not os.path.exists(respath): os.makedirs(respath)


# Load prediction arrays
print(f"\n\nFound {len(h5_files)} files matching {pattern}")

args = {"filepaths":h5_files, "group": "test", "n_subjects": test.N,
        "n_vertex": vu.vertex_info_10k[f"gray{h.lower()}"].size, "n_threads": 10}
test_obs, test_pred, test_var = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}")
_, avg_pred, avg_var = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}_avg")
_, null_pred, _ = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}_null")

print(f"\n\nTest data shape: {test_obs.shape}, {test_pred.shape}, {avg_pred.shape}, {null_pred.shape}")

fname = test.outpath(f"{test.id}.{h}.FC_embeddings.a05_t0.G{grad}.csv")
full_gradients = np.loadtxt(fname, delimiter=",")

test_means = full_gradients - test_obs

subj_IDs = np.repeat(np.arange(test.N), test_obs.shape[1]).reshape(test_obs.shape)

vtx_IDs = np.tile(np.arange(test_obs.shape[1]), test.N).reshape(test_obs.shape)

fname = test.outpath(f"test.{h}.SL_IDs.JE_cauchy{scale}_l{side}.npy")
sl_affiliation = np.load(fname)
sl_IDs = np.unique(sl_affiliation)

In [ ]:
# Compare errors between models
from variograd_utils.test_utils import compare_errors

args = {"stat_test": stat_test,
        "alternative": alternative,
        "weights": 1 - test_var / np.nanmax(test_var)}

# test against group avg
Mod_Avg = compare_errors(test_obs, test_pred, avg_pred, **args)
# Test against means
Mod_Means = compare_errors(full_gradients, test_means + test_pred, test_means, **args)
# Test against mean+avg
MeanMod_MeanAvg = compare_errors(full_gradients, test_means + test_pred, test_means + avg_pred, **args)
# test if adding avg worsens means prediction
Means_MeanAvg = compare_errors(full_gradients, test_means, test_means + avg_pred, **args)

args["weights"] = np.atleast_3d(args["weights"])
args["agg_errors"] = "mean"

# Test against null
Mod_Null = compare_errors(np.atleast_3d(test_obs),
                              np.atleast_3d(test_pred),
                              null_pred, **args)
# Tests against means + null
MeanMod_MeanNull = compare_errors(np.atleast_3d(full_gradients),
                                    np.atleast_3d(test_means + test_pred),
                                    np.atleast_3d(test_means) + null_pred, **args)
# test if adding null worsens means prediction
Means_MeanNull = compare_errors(np.atleast_3d(full_gradients),
                                    np.atleast_3d(test_means),
                                    np.atleast_3d(test_means) + null_pred, **args)


# correct for fdr
from statsmodels.stats.multitest import multipletests

p_unadj_all = np.hstack([
    Mod_Avg["pval"],
    Mod_Means["pval"],
    MeanMod_MeanAvg["pval"],
    Means_MeanAvg["pval"]
    ])

p_adj_all = multipletests(p_unadj_all, method="fdr_bh")[1]
p_adj_all = np.split(p_adj_all, 4)

Mod_Avg["pval_adj"] = p_adj_all[0]
Mod_Means["pval_adj"] = p_adj_all[1]
MeanMod_MeanAvg["pval_adj"] = p_adj_all[2]
Means_MeanAvg["pval_adj"] = p_adj_all[3]


from variograd_utils import save_hdf5
resdict = {
    "Mod_Avg": Mod_Avg,
    "Mod_Means": Mod_Means,
    "Mod_Null": Mod_Null,
    "MeanMod_MeanAvg": MeanMod_MeanAvg,
    "MeanMod_MeanNull": MeanMod_MeanNull,
    "Means_MeanNull": Means_MeanNull,
    "Means_MeanAvg": Means_MeanAvg,
}

for name, res in resdict.items():
    fname = train.outpath(f"results/VTXresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{name}.h5")
    save_hdf5(d=res, filename=fname)

print(f"\n\nSaved results to {respath}\n")

In [ ]:
# Adjust p-values across scales and models

fname = train.outpath(f"results/VTXresults.{h}.G{grad}.S{{0}}.side{side}.centered.{{1}}.h5")
scales = [10, 50, 200]
models = ["Mod_Means", "MeanMod_MeanAvg", "MeanMod_MeanNull"]
results_dict = {(scale, model): load_hdf5(fname.format(scale, model)) for scale, model in product(scales, models)}

pvals_all = {scale:
                {"Mod_Means": results_dict[(scale, "Mod_Means")]["pval"],
                 "MeanMod_MeanAvg": results_dict[(scale, "MeanMod_MeanAvg")]["pval"],
                 "MeanMod_MeanNull": np.nanmean(results_dict[(scale, "MeanMod_MeanNull")]["pval"], axis=1)}
             for scale in scales}
pvals_all = {(scale, model): pvals_all[scale][model] for scale in scales for model in models}

pvals_all = pd.DataFrame.from_dict(pvals_all).T
pvals_all = pvals_all.melt(ignore_index=False, value_vars=pvals_all.columns)

pvals_all["adj_all"] = multipletests(pvals_all["value"], method="fdr_bh")[1]

for scale, model in pvals_all.index.unique():
    results_tmp = results_dict[(scale, model)]
    results_tmp["pval_adj_all"] = pvals_all.loc[(scale, model), "adj_all"].to_numpy()
    save_hdf5(results_tmp, fname.format(scale, model))


#### Compare scales

In [ ]:
from variograd_utils import dataset, load_hdf5, print_structure
from variograd_utils.brain_utils import vertex_info_10k as vinfo
import os
import numpy as np
from itertools import combinations
import pandas as pd
from scipy.stats import ttest_rel

grad = 1
h = "R"
side = 4
mdl_name = "centered"
stat_test = "ttest_rel"
alternative = "two-sided"


train = dataset("train")
test = dataset("test")

respath = train.outpath("results")
if not os.path.exists(respath): os.makedirs(respath)

fname = train.outpath(f"results/VTXresults.{h}.G{grad}.S{{0}}.side{side}.{mdl_name}.Mod_Means.h5")

metric = "stat"

results_10 = load_hdf5(fname.format(10))
results_50 = load_hdf5(fname.format(50))
results_200 = load_hdf5(fname.format(200))

pairs = list(combinations([10, 50, 200], 2))
stats = pd.DataFrame(columns=[10, 50, 200], index=[10, 50, 200], dtype=float)
pvals = pd.DataFrame(columns=[10, 50, 200], index=[10, 50, 200], dtype=float)

for s1, s2 in pairs:
    stat1 = locals()[f"results_{s1}"][metric]
    stat2 = locals()[f"results_{s2}"][metric]

    stats.loc[s1, s2], pvals.loc[s1, s2] = ttest_rel(stat1, stat2, nan_policy="omit")
    print(f"Scale {s1} vs {s2}: stat={stats.loc[s1, s2]:.2f}, p={pvals.loc[s1, s2]:.3e}")

# Basic model

### Searchlight tests

#### Full loop

In [ ]:
# Import libraries and set parameters

import os
import glob
import numpy as np
from variograd_utils import dataset, save_hdf5
from variograd_utils.loadsave_utils import load_predictions_arrays
from variograd_utils.test_utils import compare_errors
from variograd_utils.brain_utils import vertex_info_10k as vinfo
from itertools import product

grad = 1
scale = 10
h = "L"
side = 4
mdl_name = "basic"
stat_test = "ttest_rel"
alternative = "less"

train = dataset("train")
test = dataset("test")
params = list(product([1, 2, 3], [10, 50, 200], ["L", "R"]))[4:]
for grad, scale, h in params:
    try:
        pattern = f"{train.output_dir}/LKresults.S{scale}.G{grad}/side{side}/{h}/LKresults_hdf5_G{grad}_sl*.h5"
        h5_files = glob.glob(pattern)

        respath = train.outpath("results")
        if not os.path.exists(respath): os.makedirs(respath)

        # Load prediction arrays and compute errors
        test_obs, test_pred, test_var = load_predictions_arrays(h5_files, "test", f"{mdl_name}", 
                                        n_threads=3, n_subjects=test.N,
                                        n_vertex=vinfo[f"gray{h.lower()}"].size)

        _, avg_pred, avg_var = load_predictions_arrays(h5_files, "test", f"{mdl_name}_avg", 
                                        n_threads=3, n_subjects=test.N,
                                        n_vertex=vinfo[f"gray{h.lower()}"].size)

        _, null_pred, null_var = load_predictions_arrays(h5_files, "test", f"{mdl_name}_null", 
                                        n_threads=3, n_subjects=test.N,
                                        n_vertex=vinfo[f"gray{h.lower()}"].size)


        fname = test.outpath(f"{test.id}.{h}.FC_embeddings.a05_t0.G{grad}.csv")
        avg_gradients = np.tile(np.loadtxt(fname, delimiter=",").mean(axis=0), [test_obs.shape[0], 1])

        subj_IDs = np.repeat(np.arange(test.N), test_obs.shape[1]).reshape(test_obs.shape)

        vtx_IDs = np.tile(np.arange(test_obs.shape[1]), test.N).reshape(test_obs.shape)

        fname = test.outpath(f"test.{h}.SL_IDs.JE_cauchy{scale}_l{side}.npy")
        sl_affiliation = np.load(fname)
        sl_IDs = np.unique(sl_affiliation)



        # Compare errors across different models

        Mod_Avg = {}
        Mod_AvgGeo = {}
        Mod_NullGeo = {}

        for i, sl in enumerate(sl_IDs):
            subjs = subj_IDs[sl_affiliation == sl]
            subjs_unq, n_obs = np.unique(subjs, return_counts=True)
            mask = np.isin(subjs, subjs_unq[n_obs > 1])
            
            subjs = subjs[mask]
            avg_grad = avg_gradients[sl_affiliation == sl][mask]
            true = test_obs[sl_affiliation == sl][mask]
            if np.isnan(true).all(): continue
            pred = test_pred[sl_affiliation == sl][mask]
            avg_geo = avg_pred[sl_affiliation == sl][mask]
            null_geo = null_pred[sl_affiliation == sl][mask]

            args = {"stat_test": stat_test, "alternative": alternative, "groupby": subjs}
            # Test against group avg
            Mod_Avg[sl] = compare_errors(true, pred, avg_grad, **args)
            # Test against means
            Mod_AvgGeo[sl] = compare_errors(true, pred, avg_geo, **args)
            # Test against null
            Mod_NullGeo[sl] = compare_errors(true, pred, null_geo, **args)


        resdict = {
            "Mod_Avg": Mod_Avg,
            "Mod_AvgGeo": Mod_AvgGeo,
            "Mod_NullGeo": Mod_NullGeo
        }

        for name, res in resdict.items():
            fname = train.outpath(f"results/SLresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{name}.h5")
            save_hdf5(d=res, filename=fname)

        print(f"\n\nCompleted analysis for G{grad}, S{scale}, {h}\n")
    except Exception as e:
        print(f"Error processing G{grad}, S{scale}, {h}:\n\t{e}")


#### Code sections

In [ ]:
# Import libraries and set parameters

import os
import glob
import numpy as np
from variograd_utils import dataset, save_hdf5
from variograd_utils.loadsave_utils import load_predictions_arrays
from variograd_utils.test_utils import compare_errors
from variograd_utils.brain_utils import vertex_info_10k as vinfo

grad = 2
scale = 10
h = "L"
side = 4
mdl_name = "basic"
stat_test = "ttest_rel"
alternative = "less"

train = dataset("train")
test = dataset("test")
pattern = f"{train.output_dir}/LKresults.S{scale}.G{grad}/side{side}/{h}/LKresults_hdf5_G{grad}_sl*.h5"
h5_files = glob.glob(pattern)

respath = train.outpath("results")
if not os.path.exists(respath): os.makedirs(respath)

# Load prediction arrays and compute errors

print(f"\n\nFound {len(h5_files)} files matching {pattern}")

test_obs, test_pred, test_var = load_predictions_arrays(h5_files, "test", f"{mdl_name}", 
                                  n_threads=3, n_subjects=test.N,
                                  n_vertex=vinfo[f"gray{h.lower()}"].size)

_, avg_pred, avg_var = load_predictions_arrays(h5_files, "test", f"{mdl_name}_avg", 
                                  n_threads=3, n_subjects=test.N,
                                  n_vertex=vinfo[f"gray{h.lower()}"].size)

_, null_pred, null_var = load_predictions_arrays(h5_files, "test", f"{mdl_name}_null", 
                                  n_threads=3, n_subjects=test.N,
                                  n_vertex=vinfo[f"gray{h.lower()}"].size)

print(f"\n\nTest data shape: {test_obs.shape}, {test_pred.shape}, {avg_pred.shape}, {null_pred.shape}")

fname = test.outpath(f"{test.id}.{h}.FC_embeddings.a05_t0.G{grad}.csv")
avg_gradients = np.tile(np.loadtxt(fname, delimiter=",").mean(axis=0), [test_obs.shape[0], 1])

subj_IDs = np.repeat(np.arange(test.N), test_obs.shape[1]).reshape(test_obs.shape)

vtx_IDs = np.tile(np.arange(test_obs.shape[1]), test.N).reshape(test_obs.shape)

fname = test.outpath(f"test.{h}.SL_IDs.JE_cauchy{scale}_l{side}.npy")
sl_affiliation = np.load(fname)
sl_IDs = np.unique(sl_affiliation)

print(f"\n\nFound {len(sl_IDs)} unique SLs in {fname}")


In [ ]:
# Compare errors across different models

Mod_Avg = {}
Mod_AvgGeo = {}
Mod_NullGeo = {}

for i, sl in enumerate(sl_IDs):
    subjs = subj_IDs[sl_affiliation == sl]
    subjs_unq, n_obs = np.unique(subjs, return_counts=True)
    mask = np.isin(subjs, subjs_unq[n_obs > 1])
    
    subjs = subjs[mask]
    avg_grad = avg_gradients[sl_affiliation == sl][mask]
    true = test_obs[sl_affiliation == sl][mask]
    if np.isnan(true).all(): continue
    pred = test_pred[sl_affiliation == sl][mask]
    avg_geo = avg_pred[sl_affiliation == sl][mask]
    null_geo = null_pred[sl_affiliation == sl][mask]

    args = {"stat_test": stat_test, "alternative": alternative, "groupby": subjs}
    # Test against group avg
    Mod_Avg[sl] = compare_errors(true, pred, avg_grad, **args)
    # Test against means
    Mod_AvgGeo[sl] = compare_errors(true, pred, avg_geo, **args)
    # Test against null
    Mod_NullGeo[sl] = compare_errors(true, pred, null_geo, **args)


print("\n\nDone computing statistics.")

resdict = {
    "Mod_Avg": Mod_Avg,
    "Mod_AvgGeo": Mod_AvgGeo,
    "Mod_NullGeo": Mod_NullGeo
}

for name, res in resdict.items():
    fname = train.outpath(f"results/SLresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{name}.h5")
    save_hdf5(d=res, filename=fname)

print(f"\n\nSaved results to {respath}\n")

In [ ]:
# Compare errors between models

from itertools import product, combinations
from variograd_utils import load_hdf5
from scipy.stats import ttest_rel
import pandas as pd

# scale = 50
metric = "mae"
fname = train.outpath(f"results/SLresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{{0}}.h5")

names = ["Mod_Avg", "Mod_AvgGeo"]
reslist = [load_hdf5(fname.format(name)) for name in names]
resdict = {(name, int(sl)): stats for name, dct in zip(names, reslist) for sl, stats in dct.items()}

resdf = pd.DataFrame.from_dict(resdict, orient='index')
resdf = resdf[[f"{metric}1", f"{metric}2"]].apply(np.float32)


names = ["Mod_NullGeo"]
reslist = [load_hdf5(fname.format(name)) for name in names]

resdict_null = {(name, int(sl)): stats[f"{metric}2"] for name, dct in zip(names, reslist) for sl, stats in dct.items()}
resdf_null = pd.DataFrame.from_dict(resdict_null).T
resdf_null_tall = resdf_null.reset_index(names=["comparison", "sl_ID"])
resdf_null_tall = resdf_null_tall.melt(id_vars=["comparison", "sl_ID"], value_vars=np.arange(100), var_name="permutation", value_name=f"{metric}2")
resdf_null_tall_agg = resdf_null_tall.groupby(["comparison", "sl_ID"]).agg("mean")

errors = {
    "Model": resdf.loc["Mod_Avg", f"{metric}1"],
    "Avg gradient": resdf.loc["Mod_Avg", f"{metric}2"],
    "Avg geometry": resdf.loc["Mod_AvgGeo", f"{metric}2"],
    "Null geometry": resdf_null_tall_agg.loc["Mod_NullGeo", f"{metric}2"],
}

pairs = list(combinations(errors.keys(), 2))

searchlight_pvals = pd.DataFrame(columns=errors.keys(), index=errors.keys())
searchlight_stats = pd.DataFrame(columns=errors.keys(), index=errors.keys())

for err1, err2 in pairs:
    comparison = ttest_rel(errors[err1], errors[err2], nan_policy="omit")
    stat, pval = comparison.statistic, comparison.pvalue
    searchlight_pvals.loc[err1, err2] = pval
    searchlight_stats.loc[err1, err2] = stat
    print(f"{err1} vs {err2}: stat={stat:.2f}, p={pval:.2e}")


### Vertex tests

#### Full loop

In [ ]:
# Import libraries and set parameters

import os
import glob
import warnings
import numpy as np
from itertools import product
from statsmodels.stats.multitest import multipletests
import variograd_utils as vu
from variograd_utils.test_utils import compare_errors
warnings.filterwarnings("ignore")


grads = [1, 2, 3]
scales = [10, 50, 200]
hs = ["L", "R"]
side = 4
mdl_name = "basic"
stat_test = "ttest_rel"
alternative = "less"


train = vu.dataset("train")
test = vu.dataset("test")

for grad, scale, h in product(grads, scales, hs):
    try:
        print("\n\nProcessing G{}, S{}, {}\n".format(grad, scale, h))

        pattern = f"{train.output_dir}/LKresults.S{scale}.G{grad}/side{side}/{h}/LKresults_hdf5_G{grad}_sl*.h5"
        h5_files = glob.glob(pattern)

        respath = train.outpath("results")
        if not os.path.exists(respath): os.makedirs(respath)


        # Load prediction arrays
        args = {"filepaths":h5_files, "group": "test", "n_subjects": test.N,
        "n_vertex": vu.vertex_info_10k[f"gray{h.lower()}"].size, "n_threads": 10}
        test_obs, test_pred, test_var = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}")
        _, avg_pred, avg_var = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}_avg")
        _, null_pred, _ = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}_null")

        fname = test.outpath(f"{test.id}.{h}.FC_embeddings.a05_t0.G{grad}.csv")
        avg_gradients = np.tile(np.loadtxt(fname, delimiter=",").mean(axis=0), [test_obs.shape[0], 1])

        print(f"\tTest data shape: {test_obs.shape}, {test_pred.shape}, {avg_pred.shape}, {null_pred.shape}")


        # Compare errors between models
        args = {"stat_test": stat_test,
                "alternative": alternative,
                "weights": 1 - test_var / np.nanmax(test_var)}

        # test against group avg
        Mod_Avg = compare_errors(test_obs, test_pred, avg_gradients, **args)
        # Test against means
        Mod_AvgGeo = compare_errors(test_obs, test_pred, avg_pred, **args)

        print("\tDone computing statistics for Means and Avg models.")

        args["weights"] = np.atleast_3d(args["weights"])
        args["agg_errors"] = "mean"
        # Test against null
        Mod_NullGeo = compare_errors(np.atleast_3d(test_obs),
                                     np.atleast_3d(test_pred),
                                     null_pred, **args)


        print("\tDone computing statistics for Null models.")


        # correct for fdr
        p_unadj_all = np.hstack([Mod_Avg["pval"], Mod_AvgGeo["pval"], Mod_NullGeo["pval"]])

        p_adj_all = multipletests(p_unadj_all, method="fdr_bh")[1]
        p_adj_all = np.split(p_adj_all, 3)

        Mod_Avg["pval_adj"] = p_adj_all[0]
        Mod_AvgGeo["pval_adj"] = p_adj_all[1]
        Mod_NullGeo["pval_adj"] = p_adj_all[2]
        
        print("\tDone correcting p-values for multiple comparisons.")

        resdict = {
            "Mod_Avg": Mod_Avg,
            "Mod_AvgGeo": Mod_AvgGeo,
            "Mod_NullGeo": Mod_NullGeo,
        }

        for name, res in resdict.items():
            fname = train.outpath(f"results/VTXresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{name}.h5")
            vu.save_hdf5(d=res, filename=fname)
        print(f"\tCompleted analysis for G{grad}, S{scale}, {h}\n")

    except Exception as e:
        print(f"\t\tError processing G{grad}, S{scale}, {h}:\n\t{e}")

#### Code sections

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from variograd_utils import dataset
from variograd_utils.embed_utils import diffusion_map_embedding
from variograd_utils.brain_utils import vertex_info_10k as vinfo

grad=1
scale=10
h="R"

threshold = 95

cortex = vinfo.grayl if h=="L" else vinfo.grayr + vinfo.num_meshl
cortex_slice = slice(0, vinfo.grayl.size) if h == "L" else slice(vinfo.grayl.size, None)

# Load and threshold group average FC matrix
R = np.load(dataset("train").outpath(f"train.REST_FC.10k_fs_LR.npy")
           ).astype("float32")[:, cortex][cortex, :]
R = (R + R.T) / 2
R[R < np.percentile(R, threshold, axis=0, keepdims=True)] = 0
R[R < 0] = 0

R = cosine_similarity(R.T)

embedding, V, L = diffusion_map_embedding(R, n_components=100, alpha=0.5, diffusion_time=0, random_state=0)

In [ ]:
import os
import glob
import warnings
import numpy as np
from itertools import product
from statsmodels.stats.multitest import multipletests
import variograd_utils as vu
from variograd_utils.test_utils import compare_errors

grad=1
scale=50
h="L"
side=4
mdl_name = "basic"
stat_test = "ttest_rel"
alternative = "less"

train = vu.dataset("train")
test = vu.dataset("test")


print("\n\nProcessing G{}, S{}, {}\n".format(grad, scale, h))

pattern = f"{train.output_dir}/LKresults.S{scale}.G{grad}/side{side}/{h}/LKresults_hdf5_G{grad}_sl*.h5"
h5_files = glob.glob(pattern)

respath = train.outpath("results")
if not os.path.exists(respath): os.makedirs(respath)


# Load prediction arrays
args = {"filepaths":h5_files, "group": "test", "n_subjects": test.N,
"n_vertex": vu.vertex_info_10k[f"gray{h.lower()}"].size, "n_threads": 10}
test_obs, test_pred, test_var = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}")
_, avg_pred, avg_var = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}_avg")
_, null_pred, _ = vu.load_predictions_arrays(**args, model_name = f"{mdl_name}_null")

# fname = test.outpath(f"{test.id}.{h}.FC_embeddings.a05_t0.G{grad}.csv")
# avg_gradients = np.tile(np.loadtxt(fname, delimiter=",").mean(axis=0), [test_obs.shape[0], 1])
avg_gradients = np.tile(embedding[:, grad-1], [test_obs.shape[0], 1])

print(f"\tTest data shape: {test_obs.shape}, {test_pred.shape}, {avg_pred.shape}, {null_pred.shape}")


# Compare errors between models
args = {"stat_test": stat_test,
        "alternative": alternative,
        "weights": 1 - test_var / np.nanmax(test_var)}

# test against group avg
Mod_Avg = compare_errors(test_obs, test_pred, avg_gradients, **args)
# Test against means
Mod_AvgGeo = compare_errors(test_obs, test_pred, avg_pred, **args)

print("\tDone computing statistics for Means and Avg models.")

args["weights"] = np.atleast_3d(args["weights"])
args["agg_errors"] = "mean"
# Test against null
Mod_NullGeo = compare_errors(np.atleast_3d(test_obs),
                                np.atleast_3d(test_pred),
                                null_pred, **args)


print("\tDone computing statistics for Null models.")


# correct for fdr
p_unadj_all = np.hstack([Mod_Avg["pval"], Mod_AvgGeo["pval"], Mod_NullGeo["pval"]])

p_adj_all = multipletests(p_unadj_all, method="fdr_bh")[1]
p_adj_all = np.split(p_adj_all, 3)

Mod_Avg["pval_adj"] = p_adj_all[0]
Mod_AvgGeo["pval_adj"] = p_adj_all[1]
Mod_NullGeo["pval_adj"] = p_adj_all[2]

print("\tDone correcting p-values for multiple comparisons.")

resdict = {
    "Mod_Avg": Mod_Avg,
    "Mod_AvgGeo": Mod_AvgGeo,
    "Mod_NullGeo": Mod_NullGeo,
}

# for name, res in resdict.items():
#     fname = train.outpath(f"results/VTXresults.{h}.G{grad}.S{scale}.side{side}.{mdl_name}.{name}.h5")
#     vu.save_hdf5(d=res, filename=fname)
print(f"\tCompleted analysis for G{grad}, S{scale}, {h}\n")